In [0]:
import pyspark 
from pyspark import SparkContext

In [0]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = spark._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
hub_path = "gs://modelsonpaper/processed/hub_stats"
arxiv_path = "gs://modelsonpaper/processed/arxiv"

hub_rdd = sc.textFile(hub_path).map(lambda x: x.split("\t"))
arxiv_rdd = sc.textFile(arxiv_path).map(lambda x: x.split("\t"))


In [0]:
hub_idflat = hub_rdd.flatMap(lambda x: [(i,(x[0],x[1],x[3],x[4])) for i in x[2].split(",")])
hub_idflat.take(3)

In [0]:
arxiv_rdd.take(3)

In [0]:
arxiv_rdd.filter(lambda x: len(x) != 6).count()

In [0]:
arxiv_idflat = arxiv_rdd.map(lambda x: (x[0],(x[1],x[2],x[3],x[4],x[5])))
arxiv_idflat.take(3)

In [0]:
joined_data = hub_idflat.join(arxiv_idflat)
joined_data.take(3)

In [0]:
print(hub_rdd.count())
print(hub_idflat.count())
print(arxiv_rdd.count())
print(joined_data.count())


In [0]:
hub_idflat.subtractByKey(arxiv_idflat).take(5)

In [0]:
hub_idflat.subtractByKey(arxiv_idflat).count()

In [0]:

joined_tab = joined_data.map(lambda x: (x[0],) + x[1][0] + x[1][1]).map(lambda x: "\t".join(x))

joined_tab.take(5)


In [0]:

joined_tab.saveAsTextFile("gs://modelsonpaper/processed/joined_data")